In [2]:
!mkdir -p ejemplo-contador

In [4]:
import os
os.chdir("/media/notebooks/ejemplo-contador")
! pwd

/media/notebooks/ejemplo-contador


In [5]:
#creamos el fichero con las frases
! echo "este fichero es una prueba de contador de palabras" >> prueba.txt
! echo "las palabras se contaran segun su numero de apariciones en este fichero" >> prueba.txt

Fase MAP. Conteno de palabras y creación de los pares <palabra, 1>.

La línea `%%writefile mapper.py` es para que el texto a partir de ella se escriba en un fichero denominado mapper.py y se guarde en el directorio donde nos encontremos

In [7]:
%%writefile mapper.py
#!/usr/bin/env python3

import sys

# entrada estandar STDIN
for linea in sys.stdin:
  # eliminamos espacios
  linea = linea.strip()
  # dividimos cada linea en palabras
  palabras = linea.split()

  # cada palabra la escribimos junto con un 1 y esta es la entrada del reducer
  for palabra in palabras:
    print ('%s\t%s' % (palabra, 1))

Writing mapper.py


Crearemos ahora otro para el `reducer.py`

In [8]:
%%writefile reducer.py
#!/usr/bin/env python3
from collections import defaultdict
import sys
  
#ulizamos un diccionario para las palabras con un entero como valor
contador_palabras = defaultdict(int)

#por cada linea contamos el numero total de apariciones
for linea in sys.stdin:
    palabra, numero = linea.strip().split('\t')
    contador_palabras[palabra] += int(numero)

# escribimos por cada palabra el numero total de apariciones
for palabra in sorted(contador_palabras.keys()):
    print ('%s\t%s' % (contador_palabras[palabra], palabra))  

Writing reducer.py


In [9]:
!ls 

mapper.py  prueba.txt  reducer.py


Ejecutamos los programas para comprobar que funcionan

In [10]:
#ejecutamos nuestro map-reduce y vemos el resultado
! cat /media/notebooks/ejemplo-contador/prueba.txt | python3 mapper.py | sort | python3 reducer.py > salida-contador

! cat salida-contador

1	apariciones
1	contador
1	contaran
3	de
1	en
1	es
2	este
2	fichero
1	las
1	numero
2	palabras
1	prueba
1	se
1	segun
1	su
1	una


### Ejemplo de Shakespeare

In [17]:
!pwd

/media/notebooks/ejemplo-contador


In [19]:
# Descargamos todas las obras de Shakespeare y limpiamos un poco el documento para guardarlo en obras-shakespeare.txt
! wget http://www.gutenberg.org/files/100/100-0.txt -O /media/notebooks/ejemplo-contador/100-0.txt

# Eliminamos caracteres no deseados
! cat 100-0.txt | sed -e 's/^\xEF\xBB\xBF//' | tr -d '.,;:?!(&_*-' > obras-shakespeare.txt

--2025-11-14 18:29:46--  http://www.gutenberg.org/files/100/100-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 152.19.134.47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.gutenberg.org/files/100/100-0.txt [following]
--2025-11-14 18:29:46--  https://www.gutenberg.org/files/100/100-0.txt
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5422721 (5.2M) [text/plain]
Saving to: ‘/media/notebooks/ejemplo-contador/100-0.txt’

/media/notebooks/ej 100%[===================>]   5.17M  5.98MB/s    in 0.9s    

2025-11-14 18:29:48 (5.98 MB/s) - ‘/media/notebooks/ejemplo-contador/100-0.txt’ saved [5422721/5422721]



In [20]:
#ejecutamos nuestro map-reduce
!cat /media/notebooks/ejemplo-contador/obras-shakespeare.txt | python3 mapper.py | sort | python3 reducer.py > contador-obras-shakespeare

#mostramos algunos ejemplos de salida
!head -8000 contador-obras-shakespeare | tail -10

2	Repose
1	Reposing
1	Reprieve
4	Reproach
2	Reprove
1	Repugnant
1	Repurchased
2	Reputation
1	Requickened
2	Require


In [21]:
# creamos en nuestra carpeta una nueva para el ejemplo contador
!hdfs dfs -mkdir /ejemplo-contador

# Subimos el fichero obras-shakespeare a HDFS
!hdfs dfs -put /media/notebooks/ejemplo-contador/obras-shakespeare.txt /ejemplo-contador

# Comprobamos se han subido correctamente
!hdfs dfs -ls /ejemplo-contador

Found 1 items
-rw-r--r--   3 root supergroup    5187566 2025-11-14 18:49 /ejemplo-contador/obras-shakespeare.txt


El comando para ejecutar hadoop con los mapper y los reducer es `hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar`

- files: son los ficheros que creamos anteriormente con las funciones Map y Reduce.
- mapper: fichero para el proceso Map
- reducer: fichero para el proceso Reduce
- input: ficheros de entrada. Recordamos que deben estar en HDFS. Podemos utilizar /carpeta/* para elegir todo el contenido de ésta.
- output: carpeta de salida en HDFS


In [22]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files mapper.py,reducer.py \
-mapper mapper.py \
-reducer reducer.py \
-input /ejemplo-contador/obras-shakespeare.txt \
-output /ejemplo-contador/salida

packageJobJar: [/tmp/hadoop-unjar4855049418019987025/] [] /tmp/streamjob2847554425297903234.jar tmpDir=null
2025-11-14 18:58:44,418 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-14 18:58:44,604 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-14 18:58:45,177 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1762969692826_0001
2025-11-14 18:58:45,760 INFO mapred.FileInputFormat: Total input files to process : 1
2025-11-14 18:58:45,925 INFO mapreduce.JobSubmitter: number of splits:2
2025-11-14 18:58:46,100 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1762969692826_0001
2025-11-14 18:58:46,100 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-11-14 18:58:46,341 INFO conf.Configuration: resource-types.xml not found
2025-11-14 18:58:46,341 INFO resource.ResourceUtils

In [23]:
# Comprobamos los ficheros de salida
!hdfs dfs -ls /ejemplo-contador/salida

Found 2 items
-rw-r--r--   3 root supergroup          0 2025-11-14 18:59 /ejemplo-contador/salida/_SUCCESS
-rw-r--r--   3 root supergroup     450113 2025-11-14 18:59 /ejemplo-contador/salida/part-00000


In [24]:
# Visualizamos una muestra del fichero de salida
! hdfs dfs -cat /ejemplo-contador/salida/part-00000  | head -8000 | tail -10

2	Repose
1	Reposing
1	Reprieve
4	Reproach
2	Reprove
1	Repugnant
1	Repurchased
2	Reputation
1	Requickened
2	Require
cat: Unable to write to output stream.
